In [81]:
import polars as pl

In [82]:
metadata = pl.read_parquet('../database/metadata/*.parquet')

In [83]:
metadata.describe()

statistic,numberPage,pubName,name,artType,pubDate,artCategory,pdfPage,editionNumber,id,idOficio,artClass,artSize,artNotes,highlightType,highlightPriority,highlight,idMateria,highlightimage,highlightimagename
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""","""1262498""","""1262498""","""1262498""","""1262498""","""1262498""","""1262498""","""1262498""","""1262495""","""1262498""","""218265""","""218265""","""218265""","""218265""","""213912""","""213912""","""217717""","""216936""","""145833""","""145833"""
"""null_count""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""3""","""0""","""1044233""","""1044233""","""1044233""","""1044233""","""1048586""","""1048586""","""1044781""","""1045562""","""1116665""","""1116665"""
"""mean""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""1""","""DO1""","""# EXTRATO DE 2TA 004CABW2016""","""#AVISO""","""01/01/2003""","""""","""http://pesquisa.in.gov.br/impr…","""-""","""1020_20190116_11391945""","""10000001""","""00005:00000:00000:00000:00000:…","""12""","""""","""""","""""","""""","""10449257""","""""",""""""
"""25%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""99-100-101-329""","""DO3E""","""“DECISÃO""","""“DECISÃO""","""31/12/2025""","""·048H - Aumentar a proporção d…","""http://pesquisa.in.gov.br/impr…","""99""","""701_20230101_20225293-2.xml""","""9999978""","""00099:00016:00037:00523:00000:…","""8""","""É vedada a publicação no DOU d…","""Destaques Do Diário Oficial da…","""3""","""tit""","""23634373""","""data:image/jpeg;base64,/9j/4AA…","""3_MD_14344351_001.jpg"""


In [84]:
metadata = metadata.with_columns(
    id = pl.when(pl.col('id').str.ends_with('.xml')).then(pl.col('id').str.replace('.xml', '')).otherwise(pl.col('id'))
)

In [85]:
print('Quantidade de dados nulos:\n')
for col in metadata.columns:
    print(col, metadata.select(col).null_count().item(), sep=': ')

Quantidade de dados nulos:

numberPage: 0
pubName: 0
name: 0
artType: 0
pubDate: 0
artCategory: 0
pdfPage: 0
editionNumber: 3
id: 0
idOficio: 1044233
artClass: 1044233
artSize: 1044233
artNotes: 1044233
highlightType: 1048586
highlightPriority: 1048586
highlight: 1044781
idMateria: 1045562
highlightimage: 1116665
highlightimagename: 1116665


Por que a quantidade de dados nulos das colunas 'idOficio', 'artClass', 'artSize' e 'artNotes' é a mesma?

Acredito que esteja relacionado com o tempo, no sentido que esses atributos passaram a ser adotados em modelos posteriores das publicações.

- Não é relacionado com o tempo. Temos nulos disstribuídos de 2002 a 2025.
- O atributo que mais parece se relacionar com a ausência desses dados é o pdfPage, porque é distribuído mais uniformemente. Basendo-se nisso, posso imaginar que esses atributos só constem na primeira publicação de determinada página, por exemplo. Assim poderemos interpolar e distribuir esses atributos pelas outras linhas que possuam a mesma pdfPage.
- Não consegui entender o motivo desses dados nulos do idOficio, mas vou tomar a decisão da simplicidade e remover os dados nulos completamente do dataset.

In [86]:
metadata = metadata.select('id', 'numberPage', 'pubName', 'name', 'artType', 'pubDate', 'artCategory', 'pdfPage', 'editionNumber')
metadata

id,numberPage,pubName,name,artType,pubDate,artCategory,pdfPage,editionNumber
str,str,str,str,str,str,str,str,str
"""12002010442""","""12""","""DO1""","""PORTARIA""","""PORTARIA""","""04/01/2002""","""Ministério da Agricultura,/Pec…","""http://pesquisa.in.gov.br/impr…","""3"""
"""12002010443""","""12""","""DO1""","""PORTARIA""","""PORTARIA""","""04/01/2002""","""Ministério da Agricultura,/Pec…","""http://pesquisa.in.gov.br/impr…","""3"""
"""12002010444""","""12""","""DO1""","""PORTARIA""","""PORTARIA""","""04/01/2002""","""Ministério da Agricultura,/Pec…","""http://pesquisa.in.gov.br/impr…","""3"""
"""12002010445""","""12""","""DO1""","""PORTARIA""","""PORTARIA""","""04/01/2002""","""Ministério da Agricultura,/Pec…","""http://pesquisa.in.gov.br/impr…","""3"""
"""12002010746""","""5""","""DO1""","""PORTARIA""","""PORTARIA""","""07/01/2002""","""Ministério da Defesa/GABINETE …","""http://pesquisa.in.gov.br/impr…","""10.397,DE28DEDEZEMBRODE2001"""
…,…,…,…,…,…,…,…,…
"""530_20251231_23469728""","""58""","""DO3""","""2025-12-30/11503645/COMPRASNET…","""Extrato de Credenciamento""","""31/12/2025""","""Ministério da Defesa/Comando d…","""http://pesquisa.in.gov.br/impr…","""249"""
"""530_20251231_23470217""","""58""","""DO3""","""2025-12-30/11503746/COMPRASNET…","""Extrato de Credenciamento""","""31/12/2025""","""Ministério da Defesa/Comando d…","""http://pesquisa.in.gov.br/impr…","""249"""
"""530_20251231_23470438""","""58""","""DO3""","""2025-12-30/11503765/COMPRASNET…","""Extrato de Credenciamento""","""31/12/2025""","""Ministério da Defesa/Comando d…","""http://pesquisa.in.gov.br/impr…","""249"""


In [87]:
metadata = (
    metadata
    .with_columns(
        pl.col('artType').replace(
            {
                'EXTRATO': 'Extrato',
                'EXTRATOS': 'Extrato',
                'AVISO': 'Aviso',
                'AVISOS': 'Aviso',
                'PORTARIA': 'Portaria', 
                'PORTARIAS': 'Portaria', 
                'PREGÃO': 'Pregão',
                'RESULTADO': 'Resultado',
                'RESULTADOS': 'Resultado',
                'RETIFICAÇÃO': 'Retificação',
                'RETIFICAÇÕES': 'Retificação',
                'EDITAL': 'Edital',
                'EDITAIS': 'Edital',
                'DESPACHO': 'Despacho',
                'DESPACHOS': 'Despacho',
                'TOMADA': 'Tomada',
                'ATA': 'Ata',
                'DECISÃO': 'Decisão',
                'PROCESSO': 'Processo',
                'PROCESSOS': 'Processo',
                'CONVITE': 'Convite',
                'RESOLUÇÃO': 'Resolução',
                'CONCORRÊNCIA': 'Concorrência',
                'DECRETO': 'Decreto',
                'LEILÃO': 'Leilão',
                'DELIBERAÇÃO': 'Deliberação',
                'EXPEDIENTE': 'Expediente',
                'ANEXO': 'Anexo',
                'ANEXOS': 'Anexo'
            }
        )
    )
)

In [88]:
metadata = (
    metadata
    .with_columns(
        artCategory = pl.col('artCategory')
        .str.to_uppercase()
        .str.replace('ª', '', literal=True)
        .str.replace('ª', '', literal=True)
        .str.replace('º', '', literal=True)
        .str.replace('º', '', literal=True)
    )
)

In [89]:
MINISTÉRIO DA DEFESA/COMANDO DO EXÉRCITO/COMANDO MILITAR DO NORDESTE/1 GRUPAMENTO DE ENGENHARIA/3º BATALHÃO DE ENGENHARIA DE CONSTRUÇÃO

SyntaxError: invalid syntax (2214526388.py, line 1)

In [ ]:
categorias = (
    metadata
    .group_by('artCategory')
    .agg(pl.col('id').count())
    .sort('id', descending=True)
)